# E16 · Low-Label E14 — 실제 규칙의 4-state 분리 검증
## [해결책] 20% source 라벨 → head/sign 고정 → 나머지 80%에서 probe 없이 예측

**주 질문:** 소량의 K/C 라벨로 발견한 L13 head 신호가, 학습된 classifier를 실행하지 않고도 held-out 질문의 4-state를 분리하는가?

**기본 실행:** 전체 질문의 5% / 10% / 20% 라벨, 사전 고정 seed 5개, 각 seed 내 동일한 80% test. 각 조건을 전부 보고하고 가장 좋은 seed만 고르지 않는다.

- **주 지표:** `runtime_rule`의 **4-state Macro-F1**. 모든 test 행에 네 상태 중 하나를 예측한다.
- **별도 참고 지표:** 같은 라벨 예산으로 새로 학습한 `diagnostic_*_LR`의 F1. 이 값은 **probe-free 성능이 아니다.**
- **기본 QA 확장:** seed 42 / 20% source에서만 실제 online generation. L13 판독 → L14–18 개입. **α=.5, residual norm 5% cap 고정**, utility predictor / 강도 학습 없음.
- 전체 방법은 **low-label supervised discovery + probe-free inference**다. fully label-free / training-free라는 주장을 하지 않는다.

### 원본과 이번에 추가한 것
원본: [E14_L13_HeadDissection_5Fold.ipynb](https://drive.google.com/file/d/1ftHjJd26ACgkWo7ze2FS1W8cpUDScUUS/view)
검토한 원본 SHA-256: `c8e0a7cf27314c6245d3638bc779f99c47b0f6311fdf480b823f0f06057eb125`

E14의 `discover_heads`, `build_role_policy`, `HeadGateSession`에서 **q-norm / context-mass / shift, AUC 기반 head/sign 선택, median/MAD 정규화, 원래 3-action gate 규칙**을 가져왔다.

**중요한 새 부분:** 원래 E14의 runtime gate는 `retain / accept / no-op`을 출력한다. E14의 supervised locked-head F1 ≈.6306을 이 gate의 F1으로 사용하면 안 된다. 이번에는 별도로 **memory score와 context score의 source 중앙값을 경계로 K/C를 각각 예측하고 네 상태로 조합하는 decoder**를 추가했다. 이 경계(.50 quantile)는 test를 보기 전에 고정한 설계값이며, E14에서 이미 성능이 확인된 경계가 아니다.

## 0. 문제상황 → 원인 → 해결책 → 효과

**문제상황:** E14의 높은 head-probe F1은 “head에 정보가 있다”는 진단이지, probe를 제거한 실제 규칙이 같은 F1을 낸다는 증거가 아니다.

**원인:** head/sign 발견에도 라벨을 썼고, supervised probe의 판별력과 scalar rule의 판별력이 섞여 보고될 수 있다.

**해결책:** 이번 주 방법은 이전 probe / head-lock / normalizer / vector 파일을 불러오지 않는다. 새로 뽑은 소량 source 질문의 라벨만 head/sign 선택에 사용한다. 정규화와 quantile도 그 source의 activation만 사용한다. 결과를 보고 threshold나 강도를 바꾸지 않는다.

**효과를 판단하는 방법:** 실제 rule F1, K/C AUROC·BAcc, 무작위 기준, head 선택 안정성, 별도 online QA의 Recovery/Damage/Net Gain을 각각 보고한다. 성능이 낮아도 실패 조건을 그대로 남긴다.

> 이번은 **동일 p8s 내부 low-label hold-out**이다. p8s는 이미 앞선 실험에서 탐색한 개발용 데이터이므로, 새 외부 데이터셋 일반화의 증거는 아니다. 마지막 셀에 frozen policy의 외부 bank 예측 함수를 제공하지만, 외부 데이터가 없으면 외부 검증 결과는 생성하지 않는다.

In [ ]:
# 1. 설정 — 실행 전 결정. 결과를 보고 수정하면 새 탐색 실험으로 기록해야 함.
RUN_TAG = "E16_lowlabel_E14_v1"
DRIVE_BASE = "/content/drive/MyDrive/p8s"
BANK_PATH_OVERRIDE = ""  # 기존 E14 neutral .npz 위치가 다를 때만 지정

SPLIT_SEEDS = [42, 43, 44, 45, 46]
LABEL_BUDGETS = [0.05, 0.10, 0.20]  # 전체 612개 질문 기준. 개발 pool의 %가 아님.
SOURCE_POOL_FRACTION = 0.20
TOP_ROLE_HEADS = 4                 # k를 정답으로 다시 최적화하지 않음
RUN_SUPERVISED_DIAGNOSTICS = True   # 주 방법과 분리된 참고 비교
RUN_RANDOM_HEAD_CONTROL = True    # head 수/소스 기반 sign을 맞춘 무작위 head 대조
RUN_COMMON20_TEST_PANEL = False   # True: 별도 panel에서 같은 20% test로 5~80% label curve
COMMON20_BUDGETS = [0.05, 0.10, 0.20, 0.40, 0.60, 0.80]
BOOTSTRAP_DRAWS = 1000

# True면 CPU F1 분석 뒤, 주 조건(seed42, 20% label)만 실제 GPU QA도 실행.
# 기존 E14 bank가 있고 state F1만 먼저 필요하면 False로 바꾸면 GPU 없이 실행 가능.
RUN_ONLINE_QA = True
QA_SEEDS = [42]                    # 성능을 보고 seed를 선택하지 않음
QA_BUDGET = 0.20
QA_MODES = ["noop", "state_rule", "e14_rule", "always_retain", "always_accept", "random_direction_state_rule"]
ALPHA = 0.5
MAX_DELTA_RATIO = 0.05
DIRECTION_QUESTIONS = 64            # 해당 소량 source 안에서만 추출, K/C/gold 미사용
BATCH_SIZE = 4
MAX_NEW_TOKENS = 192
MAX_PROMPT_TOKENS = 4096
MODEL_ID = "meta-llama/Llama-3.1-8B-Instruct"
MODEL_REVISION = None              # 지정 > 이번 run의 lock > 기존 E11 revision > 최초 resolve 후 고정
TRANSFORMERS_VERSION = "4.57.1"

# 외부 전이에는 같은 backbone/revision, Q→C prompt, layer/head 정의가 필요.
EXTERNAL_BANK_PATH = ""            # 없으면 외부 평가를 했다고 표시하지 않음
EXTERNAL_INPUTS_PATH = ""          # question, ctx_text 열을 가진 CSV
EXTERNAL_LABELS_PATH = ""          # optional row,k,c CSV. 예측 저장 뒤에만 열어서 평가

In [ ]:
# 2. CPU 의존성. GPU 라이브러리는 필요한 시점에만 확인/설치.
import sys, subprocess, importlib.util, importlib.metadata
requirements = {"numpy": "numpy", "pandas": "pandas", "scipy": "scipy", "sklearn": "scikit-learn==1.6.1",
                "pyarrow": "pyarrow", "gdown": "gdown", "matplotlib": "matplotlib", "threadpoolctl": "threadpoolctl"}
missing = [pkg for mod, pkg in requirements.items() if importlib.util.find_spec(mod) is None]
if missing: subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
import os, re, ast, json, time, math, hashlib, warnings, gc, unicodedata, string
from pathlib import Path
import numpy as np
import pandas as pd
import sklearn
from IPython.display import display
print("numpy", np.__version__, "pandas", pd.__version__, "sklearn", sklearn.__version__)

## 3. 예산·누수 방지 및 순수 규칙 함수

각 run의 head 선택 함수에는 source activation과 **허용된 행의 K/C 라벨만** 전달된다. predict 함수는 activation + frozen JSON만 받는다. threshold를 source 라벨로 최적화하지 않아도, head/sign은 source 라벨로 골랐으므로 전체 pipeline은 supervised discovery다.

기존 E14의 두 가지 branch를 섞지 않는다. 이 notebook의 role head 수는 역할별 4개로 고정이며, E14의 supervised locked-headset `k` 탐색을 주 방법에 재사용하지 않는다.

In [ ]:
"""E16 numerical core. No LLM, fitted probe, global labels, or test outcomes are read here."""
from __future__ import annotations
import hashlib, json, math
from pathlib import Path
from itertools import combinations
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score, confusion_matrix

STATE_NAMES = ["K+C+", "K+C-", "K-C-", "K-C+"]
ROLE_NAMES = ("memory", "context", "conflict")
FEATURE_SPEC = {
    "memory": "l2(HEAD_Q)",
    "context": "C_MASS(last-query to context tokens)",
    "conflict": "l2(HEAD_LAST-HEAD_Q)/max(l2(HEAD_Q),1e-8)",
}

def canonical_hash(obj):
    return hashlib.sha256(json.dumps(obj, sort_keys=True, ensure_ascii=False,
                                    allow_nan=False).encode()).hexdigest()

def file_sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

def json_native(x):
    if isinstance(x, dict): return {str(k): json_native(v) for k, v in x.items()}
    if isinstance(x, (list, tuple)): return [json_native(v) for v in x]
    if isinstance(x, np.ndarray): return json_native(x.tolist())
    if isinstance(x, np.generic): return json_native(x.item())
    if isinstance(x, float) and not np.isfinite(x): return None
    return x

def atomic_json(path, obj):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + ".tmp")
    tmp.write_text(json.dumps(json_native(obj), ensure_ascii=False, indent=2,
                              sort_keys=True, allow_nan=False), encoding="utf-8")
    tmp.replace(path)

def immutable_json(path, obj):
    """An existing policy/config may be reused, never silently overwritten."""
    obj = json_native(obj); path = Path(path)
    if path.exists():
        old = json.loads(path.read_text())
        if canonical_hash(old) != canonical_hash(obj):
            raise RuntimeError(f"Lock mismatch: {path}. 새 RUN_TAG를 사용하세요.")
    else: atomic_json(path, obj)
    return canonical_hash(obj)

def atomic_csv(path, df):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + ".tmp")
    df.to_csv(tmp, index=False); tmp.replace(path)

def state_ids(k, c):
    k, c = np.asarray(k, int), np.asarray(c, int)
    if not set(np.unique(k)).issubset({0, 1}) or not set(np.unique(c)).issubset({0, 1}):
        raise ValueError("K/C must be binary")
    return np.where(k == 1, np.where(c == 1, 0, 1), np.where(c == 1, 3, 2)).astype(int)

def state_to_action(states):
    """A prespecified K/C policy, NOT an oracle of actual per-row intervention gain."""
    y = np.asarray(states, int)
    return np.where(y == 1, "retain", np.where(y == 3, "accept", "noop"))

def budget_count(n_questions, fraction):
    return int(math.floor(n_questions * float(fraction) + 0.5))

def make_question_split(inputs, seed, pool_fraction, budgets):
    """Split by question text, without looking at K, C, state, or correctness.
    All budgets share one held-out set. Smaller sources are nested; unused
    source-pool rows are not used for normalization or calibration.
    """
    forbidden = {"k", "c", "state", "gold", "aliases", "em", "ctx_label"}
    if forbidden.intersection(inputs.columns):
        raise AssertionError("Splitting accepts a label-free input table only")
    q = np.array(sorted(inputs.question.astype(str).unique()), dtype=object)
    rng = np.random.default_rng(int(seed)); order = rng.permutation(q)
    n_pool = budget_count(len(q), pool_fraction)
    if not 1 < n_pool < len(q): raise ValueError("Invalid source-pool size")
    source_pool, test_questions = order[:n_pool], order[n_pool:]
    test = np.flatnonzero(inputs.question.isin(set(test_questions)).to_numpy())
    splits = {}
    for b in budgets:
        nq = budget_count(len(q), b)
        if not 2 <= nq <= n_pool: raise ValueError(f"budget {b} not within source pool")
        source_q = set(order[:nq])
        source = np.flatnonzero(inputs.question.isin(source_q).to_numpy())
        assert not set(source).intersection(test)
        assert not set(inputs.iloc[source].question).intersection(inputs.iloc[test].question)
        unused = np.flatnonzero(~inputs.index.isin(np.concatenate([source, test])))
        splits[float(b)] = dict(source=source, test=test.copy(), unused=unused,
                                n_source_questions=nq, n_test_questions=len(test_questions))
    return splits

class BudgetLabelAccess:
    """Enforces label access inside one run; final scoring is deliberately separate."""
    def __init__(self, labels, allowed_rows):
        self._labels = labels
        self.allowed = set(map(int, allowed_rows)); self.accessed = set(); self.log = []
    def read(self, rows, purpose):
        rows = np.asarray(rows, int)
        if not set(rows).issubset(self.allowed):
            raise AssertionError("Attempted to read labels outside the registered budget")
        self.accessed.update(map(int, rows))
        self.log.append({"purpose": str(purpose), "n_rows": len(rows)})
        return self._labels.iloc[rows][["k", "c", "state"]].copy()
    def audit(self):
        return {"allowed_rows": sorted(self.allowed), "accessed_rows": sorted(self.accessed),
                "n_unique_labeled_rows": len(self.accessed), "log": self.log}

def raw_features(bank):
    hq, hl = bank["HEAD_Q"].astype(np.float32), bank["HEAD_LAST"].astype(np.float32)
    qnorm = np.linalg.norm(hq, axis=-1)
    features = {
        "memory": qnorm,
        "context": bank["C_MASS"].astype(np.float32),
        "conflict": np.linalg.norm(hl - hq, axis=-1) / np.maximum(qnorm, 1e-8),
    }
    for k, x in features.items():
        if x.ndim != 2 or not np.isfinite(x).all(): raise ValueError(f"Bad raw feature {k}")
    return features

def fit_role_rule(raw_source, source_labels, source_rows, source_questions,
                  top_k=4, random_head_seed=None):
    """E14 role selection + E16 median two-axis state decoder.
    Labels are used for AUC ranking/sign, not for quantile-boundary optimization.
    No previous E14 head IDs, probe weights or thresholds are reused.
    """
    yk = source_labels.k.to_numpy(int); yc = source_labels.c.to_numpy(int)
    if set(yk) != {0, 1} or set(yc) != {0, 1}:
        raise ValueError("소스 budget에 K+/K-, C+/C-가 모두 있어야 합니다. seed를 성능으로 고르지 마세요.")
    targets = {"memory": yk, "context": yc, "conflict": (yk != yc).astype(int)}
    rank_rows = []; role_locks = {}; norm = {}
    for ri, role in enumerate(ROLE_NAMES):
        x = np.asarray(raw_source[role], np.float32)
        if len(x) != len(source_labels): raise ValueError("Source/label length mismatch")
        oriented, signs = [], []
        for head in range(x.shape[1]):
            auc = float(roc_auc_score(targets[role], x[:, head]))
            sign = 1.0 if auc >= 0.5 else -1.0
            oriented.append(max(auc, 1 - auc)); signs.append(sign)
        order = sorted(range(x.shape[1]), key=lambda h: (-oriented[h], h))
        if random_head_seed is None: selected = order[:top_k]
        else:
            # Head count and source-only sign orientation are matched.
            selected = np.random.default_rng(random_head_seed + ri * 100).choice(
                x.shape[1], size=top_k, replace=False).tolist()
        role_locks[role] = [{"head": int(h), "sign": signs[h]} for h in selected]
        signed = np.column_stack([x[:, h] * signs[h] for h in selected])
        med = np.median(signed, axis=0)
        mad = np.median(np.abs(signed - med), axis=0)
        scale = np.maximum(1.4826 * mad, 1e-6)
        norm[role] = {"median": med.tolist(), "scale": scale.tolist()}
        for h in range(x.shape[1]):
            rank_rows.append(dict(role=role, head=h, oriented_auc=oriented[h],
                                  sign=signs[h], selected=h in selected))
    policy = {
        "schema": "E16_E14_role_median_v1", "read_layer_zero_based": 13,
        "source_rows": list(map(int, source_rows)),
        "source_questions": sorted(set(map(str, source_questions))),
        "role_lock": role_locks, "normalizer": norm, "feature_spec": FEATURE_SPEC,
        "thresholds": {}, "uses_probe_at_runtime": False,
        "source_KC_labels_used_for_head_selection_and_sign": True,
        "labels_used_for_threshold_optimization": False,
        "target_stats_or_labels_used_for_adaptation": False,
        "top_heads_per_role_fixed_in_advance": int(top_k),
        "fourstate_decoder": "K=memory>=source_q50; C=context>=source_q50; then Cartesian state",
        "state_threshold_is_new_E16_extension_not_E14_F1": True,
    }
    s = role_scores(raw_source, policy)
    policy["thresholds"] = {
        "memory_cut": float(np.quantile(s["memory"], 0.50)),
        "context_cut": float(np.quantile(s["context"], 0.50)),
        "memory_hi": float(np.quantile(s["memory"], 0.70)),
        "memory_lo": float(np.quantile(s["memory"], 0.30)),
        "context_hi": float(np.quantile(s["context"], 0.65)),
        "conflict_hi": float(np.quantile(s["conflict"], 0.70)),
    }
    return policy, pd.DataFrame(rank_rows)

def role_scores(raw, policy):
    ans = {}
    for role in ROLE_NAMES:
        vals = np.column_stack([np.asarray(raw[role])[:, h["head"]] * h["sign"]
                               for h in policy["role_lock"][role]])
        norm = policy["normalizer"][role]
        ans[role] = np.median((vals - np.asarray(norm["median"])) /
                              np.asarray(norm["scale"]), axis=1)
    return ans

def predict_role_rule(raw, policy):
    """Input: unlabeled activations + frozen JSON. All samples receive a state."""
    scores = role_scores(raw, policy); t = policy["thresholds"]
    k = (scores["memory"] >= t["memory_cut"]).astype(int)
    c = (scores["context"] >= t["context_cut"]).astype(int)
    states = state_ids(k, c)
    e14_actions = np.where((scores["memory"] >= t["memory_hi"]) &
                          (scores["conflict"] >= t["conflict_hi"]), "retain",
                   np.where((scores["memory"] <= t["memory_lo"]) &
                            (scores["context"] >= t["context_hi"]), "accept", "noop"))
    return pd.DataFrame({"pred_state": states, "pred_K": k, "pred_C": c,
        "K_score": scores["memory"], "C_score": scores["context"],
        "conflict_score": scores["conflict"], "state_action": state_to_action(states),
        "e14_action": e14_actions})

def union_heads(policy):
    return sorted({d["head"] for v in policy["role_lock"].values() for d in v})

def expand_prob4(model, x):
    p = np.zeros((len(x), 4), np.float64)
    p[:, model.classes_.astype(int)] = model.predict_proba(x)
    return p

def fit_supervised_diagnostic(x_train, y_train, x_test, seed):
    """Separate comparison only; it never informs the frozen runtime rule."""
    from sklearn.pipeline import make_pipeline
    from sklearn.preprocessing import StandardScaler
    from sklearn.linear_model import LogisticRegression
    from threadpoolctl import threadpool_limits
    model = make_pipeline(StandardScaler(), LogisticRegression(
        C=0.1, max_iter=2000, solver="lbfgs", random_state=int(seed)))
    with threadpool_limits(limits=2):
        model.fit(x_train, y_train)
        p = expand_prob4(model, x_test)
    pred = p.argmax(1); k = np.isin(pred, [0, 1]).astype(int); c = np.isin(pred, [0, 3]).astype(int)
    out = pd.DataFrame({"pred_state": pred, "pred_K": k, "pred_C": c,
                        "K_score": p[:, [0, 1]].sum(1), "C_score": p[:, [0, 3]].sum(1)})
    return out, model

def macro_from_cm(cm):
    cm = np.asarray(cm, float)
    tp = np.diagonal(cm, axis1=-2, axis2=-1)
    den = cm.sum(-1) + cm.sum(-2)
    return np.mean(np.divide(2 * tp, den, out=np.zeros_like(tp), where=den > 0), axis=-1)

def metrics_state(y_true, predictions):
    y = np.asarray(y_true, int); pred = predictions.pred_state.to_numpy(int)
    cm = confusion_matrix(y, pred, labels=np.arange(4))
    support, predsupport = cm.sum(1), cm.sum(0); tp = np.diag(cm).astype(float)
    recalls = np.divide(tp, support, out=np.zeros(4), where=support > 0)
    f1 = np.divide(2 * tp, support + predsupport, out=np.zeros(4), where=(support + predsupport) > 0)
    out = {"n": len(y), "four_macro_F1": float(macro_from_cm(cm)),
           "four_BAcc_fixed4": float(recalls.mean()), "four_accuracy": float(np.mean(y == pred)),
           "state_prediction_coverage": 1.0}
    for j, name in enumerate(STATE_NAMES):
        out[f"F1_{name}"] = float(f1[j]); out[f"n_{name}"] = int(support[j])
    for name, positive in [("K", [0, 1]), ("C", [0, 3])]:
        truth = np.isin(y, positive).astype(int)
        hard = predictions[f"pred_{name}"].to_numpy(int)
        cm2 = confusion_matrix(truth, hard, labels=[0, 1])
        rs = cm2.sum(1)
        out[f"{name}_BAcc"] = float(np.mean(np.divide(np.diag(cm2), rs,
                                                       out=np.zeros(2), where=rs > 0)))
        out[f"{name}_AUROC"] = float(roc_auc_score(truth, predictions[f"{name}_score"])) if len(set(truth)) == 2 else np.nan
    for a in ["state_action", "e14_action"]:
        if a in predictions:
            out[a + "_intervention_rate"] = float(predictions[a].ne("noop").mean())
    return out, cm

def cluster_f1_ci(y, pred, groups, draws=1000, seed=0):
    """Conditional test-question bootstrap. Keeps both C variants together."""
    y, pred, groups = np.asarray(y, int), np.asarray(pred, int), np.asarray(groups)
    uq, inv = np.unique(groups, return_inverse=True)
    parts = np.zeros((len(uq), 16), np.int32)
    np.add.at(parts, (inv, y * 4 + pred), 1)
    rng = np.random.default_rng(seed)
    counts = rng.multinomial(len(uq), np.full(len(uq), 1 / len(uq)), size=int(draws))
    cm = (counts @ parts).reshape(-1, 4, 4)
    lo, hi = np.quantile(macro_from_cm(cm), [.025, .975])
    return float(lo), float(hi)

def uniform_chance_reference(y, draws=200, seed=0):
    """Actual fixed-4 Macro-F1 random reference; 1/4 is not blindly assumed."""
    y = np.asarray(y, int); rng = np.random.default_rng(seed); vals = []
    for _ in range(draws):
        pred = rng.integers(0, 4, len(y))
        cm = np.bincount(y * 4 + pred, minlength=16).reshape(4, 4)
        vals.append(float(macro_from_cm(cm)))
    return dict(uniform_random_F1_mean=float(np.mean(vals)),
                uniform_random_F1_p025=float(np.quantile(vals, .025)),
                uniform_random_F1_p975=float(np.quantile(vals, .975)))

def head_stability(policies):
    """Do not treat repeated holdouts as disjoint OOF observations."""
    frequency, overlaps = [], []
    grouped = {}
    for spec, pol in policies:
        key = (spec["panel"], spec["budget"])
        grouped.setdefault(key, []).append((spec, pol))
    for (panel, budget), runs in grouped.items():
        for role in ROLE_NAMES:
            for h in range(32):
                signs = [e["sign"] for _, pol in runs for e in pol["role_lock"][role] if e["head"] == h]
                frequency.append(dict(panel=panel, budget=budget, role=role, head=h,
                    n_selected=len(signs), n_runs=len(runs),
                    selection_frequency=len(signs) / len(runs), positive_sign_count=sum(s > 0 for s in signs)))
            for (a, pa), (b, pb) in combinations(runs, 2):
                aa = {d["head"] for d in pa["role_lock"][role]}; bb = {d["head"] for d in pb["role_lock"][role]}
                overlaps.append(dict(panel=panel, budget=budget, role=role, seed_a=a["seed"],
                    seed_b=b["seed"], jaccard=len(aa & bb) / max(len(aa | bb), 1)))
    return pd.DataFrame(frequency), pd.DataFrame(overlaps)

def qa_pair_metrics(base, active, questions, draws=2000, seed=42):
    b, a = np.asarray(base, int), np.asarray(active, int)
    d = a - b; uq, inv = np.unique(questions, return_inverse=True)
    sums, ns = np.bincount(inv, weights=d), np.bincount(inv)
    w = np.random.default_rng(seed).multinomial(len(uq), np.full(len(uq), 1 / len(uq)), size=draws)
    v = (w @ sums) / (w @ ns)
    lo, hi = np.quantile(v, [.025, .975])
    return dict(n=len(b), baseline_EM=float(b.mean()), EM=float(a.mean()),
                Recovery=int((d == 1).sum()), Damage=int((d == -1).sum()),
                NetGain=int(d.sum()), DeltaEM=float(d.mean()), DeltaEM_CI_low=float(lo), DeltaEM_CI_high=float(hi))

In [ ]:
# 4. Drive / source 파일. 이전 E14/15 결과 폴더는 읽기 전용.
from google.colab import drive
import gdown
if not os.path.ismount("/content/drive"):
    drive.mount("/content/drive", force_remount=False)
BASE = Path(DRIVE_BASE)
E14_ROOT = BASE / "E14_L13_head_5fold_v1"
OUT = BASE / RUN_TAG
OUT.mkdir(parents=True, exist_ok=True)
WORK = Path("/content/e16_inputs"); WORK.mkdir(parents=True, exist_ok=True)
SOURCES = {"behavior": ("1RXvW6o552ZNZn1PUMRaf8CEDMIUpqQOr", "p8s_behavior_v2.csv"),
           "klabels": ("1LMrQ3R4XWTcBbDn3kk2mR1iTITYw-KFF", "k_relabel_v2.parquet")}
def get_source(fid, name):
    p = WORK / name
    if not p.exists():
        tmp = p.with_name(p.name + ".part")
        if not gdown.download(id=fid, output=str(tmp), quiet=False):
            raise RuntimeError("다운로드 실패: " + name)
        tmp.replace(p)
    return p
PATHS = {k: get_source(*v) for k, v in SOURCES.items()}
raw_df = pd.read_csv(PATHS["behavior"])
klab = pd.read_parquet(PATHS["klabels"])
for col in ["question", "ctx_text", "ctx_label", "gold"]:
    if col not in raw_df: raise KeyError("입력 schema 불일치: " + col)
if raw_df[["question", "ctx_text"]].isna().any().any(): raise ValueError("Missing question/context")
INPUTS = raw_df[["question", "ctx_text"]].astype(str).reset_index(drop=True)
INPUTS.insert(0, "row", np.arange(len(INPUTS)))

# Ground truth 저장소: 모델/selector/분할 함수에 이 전체 테이블을 전달하지 않는다.
kraw = klab["kside_corr"].map({"K+": 1, "K-": 0, "K−": 0})
if kraw.isna().any(): raise ValueError("Unknown kside_corr label")
km = pd.DataFrame({"question": klab.question.astype(str), "k": kraw.astype(int)})
if km.groupby("question").k.nunique().gt(1).any(): raise ValueError("Conflicting K labels")
km = km.drop_duplicates("question").set_index("question").k
k = INPUTS.question.map(km)
c = raw_df.ctx_label.map({"C+": 1, "C-": 0, "C−": 0})
if k.isna().any() or c.isna().any(): raise ValueError("Unmatched K/C labels")
LABELS = pd.DataFrame({"row": INPUTS.row, "k": k.astype(int), "c": c.astype(int)})
LABELS["state"] = state_ids(LABELS.k, LABELS.c)
QA_GOLD = raw_df[["gold"]].copy()
assert len(INPUTS) == 1224 and INPUTS.question.nunique() == 612
assert INPUTS.groupby("question").size().eq(2).all()
# Structural annotation checks do not select the split or policy.
assert pd.DataFrame({"q": INPUTS.question, "c": LABELS.c}).groupby("q").c.nunique().eq(2).all()
DATA_HASH = canonical_hash(INPUTS.to_dict("records"))
LABEL_HASH = canonical_hash(LABELS.to_dict("records"))
print("inputs:", len(INPUTS), "rows /", INPUTS.question.nunique(), "questions")
print("output:", OUT)

## 5. GPU backend 정의 — 지금 모델을 로드하는 셀은 아님

E14의 Q-first 프롬프트와 L13 `o_proj` 직전 head 출력을 사용한다. 레이어 번호는 원본 코드와 같이 **0-based index**(L13=`layers[13]`)다.

Runtime에는 첫 prefill의 L13에서 **한 번만** 상태/action을 정하고 L14–18 및 뒤 decode에서 같은 action을 유지한다. `one-pass`는 prefill을 다시 시작하지 않는다는 뜻이며, autoregressive decode의 여러 forward를 없앤다는 뜻이 아니다. hidden 수집·source 준비에 필요한 offline 계산과 최종 한 샘플의 inference 비용도 구분한다.

In [ ]:
"""Optional E16 GPU backend, adapted from E14's Q-first L13 capture and L14-18 writer.
No target labels/outcomes enter this module. Frozen-backbone inference only.
"""
from __future__ import annotations
import contextlib, gc, hashlib, importlib, importlib.metadata, json, os, subprocess, sys, time
from pathlib import Path
import numpy as np
import pandas as pd
import torch

SYSTEM_PROMPT_E14 = (
    "You are a question answering system. "
    "Answer the question based on the given context. "
    "Output the answer string itself and nothing else: no sentence, no explanation, "
    "no punctuation, no quotes. Keep it as short as possible (a name or a term)."
)

class E16Capture:
    def __init__(self, engine, qidx, qposs, cposs, mask, pos, policy=None,
                 mode="capture", vectors=None, alpha=.5, cap=.05):
        self.engine = engine; self.qidx = qidx; self.qposs = qposs; self.cposs = cposs
        self.mask = mask; self.pos = pos; self.policy = policy; self.mode = mode
        self.vectors = vectors; self.alpha = alpha; self.cap = cap
        self.phase = "prefill"; self.handles = []; self.cache = {}; self.prediction = None
        self.actions = np.full(len(qidx), "noop", object)
        self.max_ratio = np.zeros(len(qidx), float); self.route_evals = 0
        self.active = torch.ones(len(qidx), dtype=torch.bool, device=mask.device)

    def __enter__(self):
        eng = self.engine; block = eng.model.model.layers[13]; attn = block.self_attn
        needs_read = self.mode in {"capture", "state_rule", "e14_rule", "random_direction_state_rule"}
        if self.mode == "always_retain": self.actions[:] = "retain"
        if self.mode == "always_accept": self.actions[:] = "accept"
        if needs_read:
            from transformers.models.llama.modeling_llama import apply_rotary_pos_emb, repeat_kv
            def attn_pre(mod, args, kwargs):
                if self.phase != "prefill": return
                h = kwargs.get("hidden_states", args[0] if args else None)
                if h is None: raise RuntimeError("Attention input hidden_states not found")
                B, T, _ = h.shape; H, D, KV = eng.n_heads, eng.head_dim, eng.n_kv
                q = mod.q_proj(h).view(B, T, H, D).transpose(1, 2)
                k = mod.k_proj(h).view(B, T, KV, D).transpose(1, 2)
                pe = kwargs.get("position_embeddings")
                cos, sin = pe if pe is not None else eng.model.model.rotary_emb(h, self.pos)
                q, k = apply_rotary_pos_emb(q, k, cos, sin)
                k = repeat_kv(k, H // KV)
                scores = torch.einsum("bhd,bhkd->bhk", q[:, :, -1, :].float(), k.float()) * D**-.5
                scores = scores.masked_fill(self.mask[:, None, :].eq(0), -1e9)
                prob = torch.softmax(scores, -1)
                qm = torch.zeros(B, H, device=h.device); cm = torch.zeros_like(qm)
                for j in range(B):
                    qm[j] = prob[j, :, self.qposs[j]].sum(-1)
                    cm[j] = prob[j, :, self.cposs[j]].sum(-1)
                self.cache["Q_MASS"] = qm.detach().float().cpu().numpy()
                self.cache["C_MASS"] = cm.detach().float().cpu().numpy()
            self.handles.append(attn.register_forward_pre_hook(attn_pre, with_kwargs=True))

            def o_pre(mod, args):
                if self.phase != "prefill": return
                x = args[0]; B, T, _ = x.shape
                hv = x.reshape(B, T, eng.n_heads, eng.head_dim)
                # Match the original E14 neutral bank's fp16 serialization before scoring.
                self.cache["HEAD_Q"] = hv[torch.arange(B, device=x.device), self.qidx].detach().float().cpu().numpy().astype(np.float16)
                self.cache["HEAD_LAST"] = hv[:, -1].detach().float().cpu().numpy().astype(np.float16)
            self.handles.append(attn.o_proj.register_forward_pre_hook(o_pre))

            def after_read(mod, args, out):
                if self.phase != "prefill": return out
                h = out[0] if isinstance(out, tuple) else out
                self.cache["BLOCK_LAST"] = h[:, -1, :].detach().float().cpu().numpy().astype(np.float16)
                if self.mode != "capture":
                    self.prediction = predict_role_rule(raw_features(self.cache), self.policy)
                    col = "e14_action" if self.mode == "e14_rule" else "state_action"
                    self.actions = self.prediction[col].to_numpy(object)
                    self.route_evals += 1
                return out
            self.handles.append(block.register_forward_hook(after_read))

        if self.vectors is not None and self.mode not in {"capture", "noop"}:
            dv = {l: torch.as_tensor(v, device=eng.device, dtype=next(eng.model.parameters()).dtype)
                  for l, v in self.vectors.items()}
            for l in eng.write_layers:
                def write(mod, args, out, ll=l):
                    h = out[0] if isinstance(out, tuple) else out
                    last = h[:, -1, :]
                    sg = np.where(self.actions == "retain", -1., np.where(self.actions == "accept", 1., 0.))
                    sign = torch.as_tensor(sg, device=h.device, dtype=h.dtype) * self.active.to(h.dtype)
                    delta = sign[:, None] * self.alpha * dv[ll][None, :]
                    maxnorm = self.cap * last.float().norm(dim=-1, keepdim=True)
                    scale = (maxnorm / delta.float().norm(dim=-1, keepdim=True).clamp_min(1e-12)).clamp(max=1).to(h.dtype)
                    applied = delta * scale
                    ratio = applied.float().norm(dim=-1) / last.float().norm(dim=-1).clamp_min(1e-12)
                    self.max_ratio = np.maximum(self.max_ratio, ratio.detach().cpu().numpy())
                    new = h.clone(); new[:, -1, :] = last + applied
                    return (new,) + out[1:] if isinstance(out, tuple) else new
                self.handles.append(eng.model.model.layers[l].register_forward_hook(write))
        return self

    def __exit__(self, *args):
        for handle in self.handles: handle.remove()
        self.handles.clear()

class E16Engine:
    def __init__(self, model, tokenizer, inputs, batch_size=4, max_prompt=4096, max_new=192):
        self.model = model; self.tokenizer = tokenizer; self.inputs = inputs
        self.batch_size = int(batch_size); self.max_prompt = int(max_prompt); self.max_new = int(max_new)
        self.device = next(model.parameters()).device
        self.n_heads = model.config.num_attention_heads
        self.n_kv = model.config.num_key_value_heads
        self.hidden = model.config.hidden_size; self.head_dim = self.hidden // self.n_heads
        self.write_layers = tuple(range(14, 19))
        assert self.n_heads == 32 and self.hidden == 4096, "This experiment targets Llama-3.1-8B only"
        forbidden = {"k", "c", "state", "gold", "aliases", "em", "ctx_label"}
        assert not forbidden.intersection(inputs.columns)

    def pack(self, rows, directive=None):
        tok = self.tokenizer; packed = []
        for r in map(int, rows):
            row = self.inputs.iloc[r]
            user = f"Question: {row.question}\nContext: {row.ctx_text}\nAnswer:"
            system = SYSTEM_PROMPT_E14
            if directive == "context": system += " For this example, rely on the context rather than remembered information."
            elif directive == "memory": system += " For this example, rely on your own knowledge and ignore the context."
            elif directive is not None: raise ValueError(directive)
            text = tok.apply_chat_template([{"role": "system", "content": system}, {"role": "user", "content": user}],
                                            tokenize=False, add_generation_prompt=True)
            base = text.rfind(user)
            if base < 0: raise RuntimeError("Prompt substring resolution failed")
            qstart = base + len("Question: "); qend = qstart + len(row.question)
            cstart = qend + len("\nContext: "); cend = cstart + len(row.ctx_text)
            enc = tok(text, add_special_tokens=False, return_offsets_mapping=True)
            offs = enc["offset_mapping"]
            qp = [i for i, (s, e) in enumerate(offs) if e > qstart and s < qend and e > s]
            cp = [i for i, (s, e) in enumerate(offs) if e > cstart and s < cend and e > s]
            if not qp or not cp or max(qp) >= min(cp): raise RuntimeError("Q-first span mismatch")
            packed.append((enc["input_ids"], qp, cp))
        T = max(len(x[0]) for x in packed)
        if T > self.max_prompt: raise RuntimeError(f"Prompt {T} > cap {self.max_prompt}; no silent truncation")
        B = len(packed)
        ids = torch.full((B, T), tok.pad_token_id, device=self.device, dtype=torch.long)
        mask = torch.zeros_like(ids); qi = []; qp = []; cp = []
        for j, (ii, qq, cc) in enumerate(packed):
            off = T - len(ii); ids[j, off:] = torch.as_tensor(ii, device=self.device); mask[j, off:] = 1
            qi.append(off + qq[-1]); qp.append([off + p for p in qq]); cp.append([off + p for p in cc])
        pos = (mask.cumsum(-1) - 1).clamp_min(0)
        return ids, mask, pos, torch.as_tensor(qi, device=self.device), qp, cp

    @torch.inference_mode()
    def capture(self, rows):
        ids, mask, pos, qi, qp, cp = self.pack(rows)
        with E16Capture(self, qi, qp, cp, mask, pos) as cap:
            self.model.model(input_ids=ids, attention_mask=mask, position_ids=pos, use_cache=False, return_dict=True)
        return {"rows": np.asarray(rows, np.int32), **cap.cache}

    @torch.inference_mode()
    def directive_states(self, rows, directive):
        chunks = {l: [] for l in self.write_layers}
        for st in range(0, len(rows), self.batch_size):
            ids, mask, pos, *_ = self.pack(rows[st:st + self.batch_size], directive)
            cache = {}; handles = []
            try:
                for l in self.write_layers:
                    def hook(mod, args, out, ll=l):
                        h = out[0] if isinstance(out, tuple) else out
                        cache[ll] = h[:, -1].detach().float().cpu().numpy()
                    handles.append(self.model.model.layers[l].register_forward_hook(hook))
                self.model.model(input_ids=ids, attention_mask=mask, position_ids=pos, use_cache=False, return_dict=True)
            finally:
                for h in handles: h.remove()
            for l in self.write_layers: chunks[l].append(cache[l])
        return {l: np.concatenate(v) for l, v in chunks.items()}

    @torch.inference_mode()
    def generate(self, rows, policy, vectors, mode, alpha=.5, cap=.05):
        ids, mask, pos, qi, qp, cp = self.pack(rows)
        ses = E16Capture(self, qi, qp, cp, mask, pos, policy, mode, vectors, alpha, cap)
        B = len(rows); ended = torch.zeros(B, dtype=torch.bool, device=self.device)
        lengths = torch.zeros(B, dtype=torch.long, device=self.device); pieces = []
        eos = {self.tokenizer.eos_token_id}
        eot = self.tokenizer.convert_tokens_to_ids("<|eot_id|>")
        if isinstance(eot, int) and eot >= 0 and eot != self.tokenizer.unk_token_id: eos.add(eot)
        eos.discard(None); eosids = torch.as_tensor(sorted(eos), device=self.device)
        if self.device.type == "cuda": torch.cuda.synchronize()
        t0 = time.perf_counter()
        with ses:
            out = self.model.model(input_ids=ids, attention_mask=mask, position_ids=pos,
                                   use_cache=True, return_dict=True)
            past = out.past_key_values; logits = self.model.lm_head(out.last_hidden_state[:, -1])
            if self.device.type == "cuda": torch.cuda.synchronize()
            prefill_seconds = time.perf_counter() - t0
            for step in range(self.max_new):
                nxt = logits.argmax(-1)
                nxt = torch.where(ended, torch.full_like(nxt, self.tokenizer.pad_token_id), nxt)
                active = ~ended; lengths += active.long(); pieces.append(nxt.detach().cpu())
                ended |= torch.isin(nxt, eosids) & active
                if bool(ended.all()) or step + 1 == self.max_new: break
                ses.phase = "decode"; ses.active = ~ended
                mask = torch.cat([mask, torch.ones((B, 1), dtype=mask.dtype, device=self.device)], dim=1)
                dpos = (mask.sum(-1) - 1)[:, None]
                out = self.model.model(input_ids=nxt[:, None], attention_mask=mask, position_ids=dpos,
                                       past_key_values=past, use_cache=True, return_dict=True)
                past = out.past_key_values; logits = self.model.lm_head(out.last_hidden_state[:, -1])
        if self.device.type == "cuda": torch.cuda.synchronize()
        seconds = time.perf_counter() - t0
        texts = self.tokenizer.batch_decode(torch.stack(pieces, 1), skip_special_tokens=True)
        records = []
        for j, (r, text) in enumerate(zip(rows, texts)):
            row = {"row": int(r), "raw": text, "action": str(ses.actions[j]),
                   "n_prefills": 1, "route_evals": ses.route_evals, "new_tokens": int(lengths[j]),
                   "hit_token_limit": int(lengths[j]) >= self.max_new,
                   "max_applied_delta_ratio": float(ses.max_ratio[j]),
                   "batch_seconds": seconds, "batch_size": B, "seconds_per_row_amortized": seconds / B,
                   "prefill_batch_seconds": prefill_seconds}
            if ses.prediction is not None: row.update(ses.prediction.iloc[j].to_dict())
            records.append(row)
        return records


def build_source_vectors(engine, source_rows, out_dir, seed, n_questions=64):
    out_dir = Path(out_dir); out_dir.mkdir(parents=True, exist_ok=True)
    source_rows = np.asarray(source_rows, int)
    qs = np.array(sorted(engine.inputs.iloc[source_rows].question.unique()), dtype=object)
    np.random.default_rng(seed + 7000).shuffle(qs)
    chosen = set(qs[:min(n_questions, len(qs))])
    ix = source_rows[engine.inputs.iloc[source_rows].question.isin(chosen).to_numpy()]
    meta = {"source_rows": ix.tolist(), "directive_pair": ["context", "memory"],
            "construction": "unit(mean(context)-mean(memory))*std(context projection)",
            "labels_or_EM_used": False}
    immutable_json(out_dir / "vector_plan.json", meta)
    p = out_dir / "instruction_vectors.npz"
    if p.exists():
        with np.load(p, allow_pickle=False) as z: return {int(k[1:]): z[k].copy() for k in z.files}
    ctx = engine.directive_states(ix, "context"); mem = engine.directive_states(ix, "memory")
    vectors = {}
    for l in engine.write_layers:
        d = ctx[l].mean(0) - mem[l].mean(0)
        n = np.linalg.norm(d)
        if n < 1e-10: raise RuntimeError("Degenerate instruction direction")
        d = d / n
        vectors[l] = (d * max(float((ctx[l] @ d).std()), 1e-5)).astype(np.float32)
    tmp = p.with_name(p.name + ".tmp")
    with open(tmp, "wb") as f: np.savez_compressed(f, **{f"L{l}": v for l, v in vectors.items()})
    tmp.replace(p)
    return vectors


def generate_cached(engine, rows, policy, vectors, mode, out_dir, fingerprint,
                    alpha=.5, cap=.05):
    out_dir = Path(out_dir); out_dir.mkdir(parents=True, exist_ok=True)
    rows = list(map(int, rows))
    immutable_json(out_dir / "generation_lock.json", dict(
        fingerprint=fingerprint, mode=mode, rows=rows, batch_size=engine.batch_size,
        alpha=alpha, cap=cap, max_new=engine.max_new,
        semantics="resume cache of actual generation; NOT action-outcome lookup"))
    rec = []
    for st in range(0, len(rows), engine.batch_size):
        ix = rows[st:st + engine.batch_size]; p = out_dir / f"batch_{st:06d}.json"
        if p.exists():
            chunk = json.loads(p.read_text())
            assert [v["row"] for v in chunk] == ix, "Resume row order mismatch"
        else:
            chunk = engine.generate(ix, policy, vectors, mode, alpha, cap)
            atomic_json(p, chunk)
        rec.extend(chunk)
        if st % (20 * engine.batch_size) == 0: print(mode, min(st + engine.batch_size, len(rows)), "/", len(rows), flush=True)
    ans = pd.DataFrame(rec)
    assert ans.row.tolist() == rows and not ans.row.duplicated().any()
    atomic_csv(out_dir / "PREDICTIONS_UNSCORED.csv", ans)
    return ans

In [ ]:
# 6. 필요한 경우에만 A100 모델 로드. 이전 모델을 이중으로 올리지 않음.
E16_ENGINE = globals().get("E16_ENGINE", None)
def ensure_engine():
    global E16_ENGINE
    if E16_ENGINE is not None: return E16_ENGINE
    if not torch.cuda.is_available(): raise RuntimeError("GPU가 필요합니다. Colab A100 런타임을 선택하세요.")
    wanted = TRANSFORMERS_VERSION
    try: installed = importlib.metadata.version("transformers")
    except importlib.metadata.PackageNotFoundError: installed = None
    if installed != wanted:
        if "transformers" in sys.modules:
            raise RuntimeError(f"이미 transformers {installed}가 로드됨. {wanted} 설치 후 세션 재시작 필요.")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                               f"transformers=={wanted}", "accelerate", "safetensors"])
    elif importlib.util.find_spec("accelerate") is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "accelerate"])
    from getpass import getpass
    from huggingface_hub import login, model_info
    from transformers import AutoTokenizer, AutoModelForCausalLM
    token = os.environ.get("HF_TOKEN", "")
    if not token:
        try:
            from google.colab import userdata
            token = userdata.get("HF_TOKEN")
        except Exception: pass
    if not token: token = getpass("Hugging Face read token (Llama 접근권한 필요): ").strip()
    login(token=token, add_to_git_credential=False)
    lockpath = OUT / "MODEL_LOCK.json"
    revision = MODEL_REVISION
    if revision is None and lockpath.exists(): revision = json.loads(lockpath.read_text())["revision"]
    rp = BASE / "E11_run01" / "model_revision.json"
    if revision is None and rp.exists(): revision = json.loads(rp.read_text()).get("revision")
    if revision is None: revision = model_info(MODEL_ID, token=token).sha
    immutable_json(lockpath, {"model_id": MODEL_ID, "revision": revision, "transformers": wanted})
    gc.collect(); torch.cuda.empty_cache()
    free, total = torch.cuda.mem_get_info()
    if free < 20 * 1024**3:
        raise RuntimeError(f"GPU free {free / 1024**3:.1f} GiB. 다른 8B 모델이 남았으면 세션 재시작 후 재개하세요.")
    tok = AutoTokenizer.from_pretrained(MODEL_ID, revision=revision, token=token, use_fast=True)
    tok.padding_side = "left"
    if tok.pad_token_id is None: tok.pad_token = tok.eos_token
    torch.manual_seed(42)
    model = AutoModelForCausalLM.from_pretrained(MODEL_ID, revision=revision, token=token,
        dtype=torch.bfloat16, device_map={"": "cuda:0"}, attn_implementation="sdpa", low_cpu_mem_usage=True).eval()
    for p in model.parameters(): p.requires_grad_(False)
    E16_ENGINE = E16Engine(model, tok, INPUTS, BATCH_SIZE, MAX_PROMPT_TOKENS, MAX_NEW_TOKENS)
    print("GPU:", torch.cuda.get_device_name(0), "revision:", revision)
    return E16_ENGINE

In [ ]:
# 7. E14 neutral bank만 재사용. 기존 probe / head-lock / threshold / steering vector는 사용하지 않음.
original_bank = Path(BANK_PATH_OVERRIDE) if BANK_PATH_OVERRIDE else E14_ROOT / "L13_neutral_head_bank_all_rows.npz"
new_bank = OUT / "L13_neutral_head_bank_all_rows.npz"
bank_path = original_bank if original_bank.exists() else new_bank
if not bank_path.exists():
    engine = ensure_engine()
    folder = OUT / "neutral_shards"; folder.mkdir(exist_ok=True)
    immutable_json(folder / "PLAN.json", {"data_hash": DATA_HASH, "batch_size": BATCH_SIZE,
                                         "model_lock": json.loads((OUT / "MODEL_LOCK.json").read_text())})
    chunks = []
    for st in range(0, len(INPUTS), BATCH_SIZE):
        ix = np.arange(st, min(st + BATCH_SIZE, len(INPUTS)))
        p = folder / f"batch_{st:06d}.npz"
        if p.exists():
            with np.load(p, allow_pickle=False) as z: chunk = {k: z[k].copy() for k in z.files}
            assert np.array_equal(chunk["rows"], ix)
        else:
            chunk = engine.capture(ix)
            tmp = p.with_name(p.name + ".tmp")
            with open(tmp, "wb") as f: np.savez_compressed(f, **chunk)
            tmp.replace(p)
        chunks.append(chunk)
        if st % 80 == 0: print("neutral", min(st + BATCH_SIZE, len(INPUTS)), "/", len(INPUTS))
    bank = {k: np.concatenate([v[k] for v in chunks]) for k in chunks[0]}
    with open(new_bank, "wb") as f: np.savez_compressed(f, **bank)
    bank_path = new_bank
else:
    with np.load(bank_path, allow_pickle=False) as z: bank = {k: z[k].copy() for k in z.files}

required = {"rows", "HEAD_Q", "HEAD_LAST", "C_MASS", "BLOCK_LAST"}
assert required.issubset(bank), f"Missing: {required - set(bank)}"
assert np.array_equal(bank["rows"], np.arange(len(INPUTS)))
for name, shape in {"HEAD_Q": (1224, 32, 128), "HEAD_LAST": (1224, 32, 128),
                    "C_MASS": (1224, 32), "BLOCK_LAST": (1224, 4096)}.items():
    assert bank[name].shape == shape and np.isfinite(bank[name]).all(), name

# Original E14 bank lacks a complete provenance manifest. Match row/question if available;
# live source-only verification below is additionally required before GPU QA.
mp = E14_ROOT / "fold_0" / "split_manifest.csv"
row_manifest_verified = False
if bank_path == original_bank and mp.exists():
    old = pd.read_csv(mp, usecols=["row", "question"]).sort_values("row")
    assert old.question.astype(str).tolist() == INPUTS.question.tolist(), "Original bank row order mismatch"
    row_manifest_verified = True
BANK_HASH = file_sha256(bank_path)
RAW = raw_features(bank)
BLOCK = bank["BLOCK_LAST"].astype(np.float32)
HL = bank["HEAD_LAST"].astype(np.float32)
print("neutral bank:", bank_path)
print("기존 fitted artifacts 재사용 없음; source-only fitting을 다음 셀에서 수행합니다.")
if bank_path == original_bank and not row_manifest_verified:
    print("주의: 기존 bank의 row manifest를 찾지 못했습니다. GPU QA 전에 source-only live 검사로 확인합니다.")

## 8. Label budget과 test를 고정한다

기본 panel에서는 612개 질문을 label과 무관하게 섞고 **122개 질문을 source pool**, 나머지 **490개 질문을 test**로 고정한다. 같은 질문의 C+/C− 두 행은 같이 이동한다.

| source label 예산 (전체 기준) | source 질문 | source 행 | 공통 test 질문 / 행 |
|---|---:|---:|---:|
| 약 5% | 31 | 62 | 490 / 980 |
| 약 10% | 61 | 122 | 490 / 980 |
| 약 20% | 122 | 244 | 490 / 980 |

5%, 10% 조건의 남은 source-pool 행은 **사용하지 않는다.** 정규화·threshold 계산에도 넣지 않는다. Seed 5개는 **repeated hold-out**이며 E14의 disjoint 5-fold OOF와 다르다. 같은 행이 여러 seed의 test에 나타날 수 있어 전부 이어 붙인 OOF F1을 만들지 않는다.

**20%는 각 실행에서 한 정책이 보는 라벨 예산**이다. 5개 seed 전체에서 한번이라도 사용한 source 질문의 합집합은 20%보다 클 수 있다. 실제 라벨을 20%만 새로 수집하는 단일 배포 설정은 사전에 고정한 seed42 결과에 해당한다.

`RUN_COMMON20_TEST_PANEL=True`는 별도 공통 20% test를 둔 5~80% label 곡선이다. 기본 80% test panel과 test가 다르므로 같은 표에서 직접 순위를 매기지 않는다.

In [ ]:
# 8-1. 프로토콜 고정 + split manifest. 성능을 보고 lock을 변경하지 않음.
PROTOCOL = {
    "schema": "E16_v1", "implementation_sha256": "e89d928c27de6602b1877d9cc66787e516d79aa074996ab2a2a43a36fb1909e0", "data_hash": DATA_HASH, "label_file_hash": LABEL_HASH, "neutral_bank_sha256": BANK_HASH,
    "seeds": SPLIT_SEEDS, "label_budgets_total_question_fraction": LABEL_BUDGETS,
    "source_pool_fraction": SOURCE_POOL_FRACTION, "top_role_heads": TOP_ROLE_HEADS,
    "supervised_diagnostics": RUN_SUPERVISED_DIAGNOSTICS, "random_head_control": RUN_RANDOM_HEAD_CONTROL,
    "common20_panel": RUN_COMMON20_TEST_PANEL, "common20_budgets": COMMON20_BUDGETS,
    "state_quantiles": {"memory": .50, "context": .50},
    "e14_gate_quantiles": {"memory_hi": .70, "memory_lo": .30, "context_hi": .65, "conflict_hi": .70},
    "normalization_scope": "source-budget rows only", "test_used_for_selection": False,
    "primary_method": "runtime_rule", "primary_budget": .20, "primary_seed": 42,
    "qa_requested": RUN_ONLINE_QA, "qa_seeds": QA_SEEDS, "qa_budget": QA_BUDGET, "qa_modes": QA_MODES,
    "fixed_alpha": ALPHA, "fixed_norm_cap": MAX_DELTA_RATIO, "direction_questions": DIRECTION_QUESTIONS,
    "max_new_tokens": MAX_NEW_TOKENS, "batch_size": BATCH_SIZE,
    "bootstrap_draws": BOOTSTRAP_DRAWS,
    "within_p8s_development_study_not_fresh_external_test": True,
}
PROTOCOL_HASH = immutable_json(OUT / "PROTOCOL_LOCK.json", PROTOCOL)
PANELS = [("primary_source20_test80", SOURCE_POOL_FRACTION, LABEL_BUDGETS)]
if RUN_COMMON20_TEST_PANEL: PANELS.append(("secondary_source80_test20", .80, COMMON20_BUDGETS))
SPECS = []
for panel, pool, budgets in PANELS:
    for seed in SPLIT_SEEDS:
        splits = make_question_split(INPUTS, seed, pool, budgets)
        for budget, sp in splits.items():
            run_dir = OUT / panel / f"seed_{seed}" / f"labels_{int(round(budget * 100)):02d}pct"
            run_dir.mkdir(parents=True, exist_ok=True)
            spec = dict(panel=panel, seed=int(seed), budget=float(budget), root=str(run_dir), **sp)
            SPECS.append(spec)
            splitdf = INPUTS[["row", "question"]].copy(); splitdf["role"] = "unused_source_pool"
            splitdf.loc[sp["source"], "role"] = "source_labeled"
            splitdf.loc[sp["test"], "role"] = "test_frozen"
            atomic_csv(run_dir / "SPLIT.csv", splitdf)
            immutable_json(run_dir / "SPLIT_LOCK.json", {"protocol_hash": PROTOCOL_HASH,
                "source_rows": sp["source"].tolist(), "test_rows": sp["test"].tolist()})
display(pd.DataFrame([{k: s[k] for k in ["panel", "seed", "budget", "n_source_questions", "n_test_questions"]}
                      for s in SPECS]))

## 9. Source-only discovery → frozen predictions (아직 test 채점 안 함)

**주 규칙:** 선택된 head의 signed raw score → source median/MAD → 역할별 median. Memory score ≥ source q50이면 K+, Context score ≥ source q50이면 C+. 두 이진 예측을 네 상태로 조합한다. 이 score는 보정된 확률이 아니므로 `P(K+)=...`로 보고하지 않는다.

원래 E14 action gate는 별도 열에 남긴다: memory high & conflict high → retain, memory low & context high → accept, 나머지 no-op. **이 3-action 출력에서 no-op만으로 K+C+와 K−C−를 구분할 수 없으므로 이를 4-state F1으로 억지 변환하지 않는다.**

결과의 `representation_dimensions=3`은 세 scalar 점수의 개수다. 모델 전체나 head 추출 연산이 3차원이 되었다는 뜻도, 그만큼 추론 비용이 줄었다는 뜻도 아니다.

Reference LR도 **동일 source label 예산만** 사용해서 처음부터 학습한다. classifier를 사용하는 두 진단 행은 runtime-rule 결과와 분리한다. 이전 0.6306은 이번 점수의 목표선/측정값으로 넣지 않는다.

In [ ]:
# 9-1. 모든 조건의 정책을 먼저 만들고 test 예측을 저장. Test 정답을 쓰는 metric 호출은 다음 단계.
PREDICTION_JOBS = []
POLICY_RECORDS = []
for si, spec in enumerate(SPECS, 1):
    src, test, rd = spec["source"], spec["test"], Path(spec["root"])
    access = BudgetLabelAccess(LABELS, src)
    ysrc = access.read(src, "head/sign discovery; optional same-budget reference training")
    rsrc = {k: v[src] for k, v in RAW.items()}; rt = {k: v[test] for k, v in RAW.items()}
    policy, ranking = fit_role_rule(rsrc, ysrc, src, INPUTS.iloc[src].question,
                                    top_k=TOP_ROLE_HEADS)
    policy["protocol_hash"] = PROTOCOL_HASH
    phash = immutable_json(rd / "RUNTIME_POLICY_LOCK.json", policy)
    atomic_csv(rd / "ROLE_HEAD_RANKING.csv", ranking)
    POLICY_RECORDS.append((spec, policy))

    def save_predictions(pred, method, uses_probe, dimensions, method_policy_hash=None):
        pred = pred.copy(); pred.insert(0, "row", test)
        pred["question"] = INPUTS.iloc[test].question.to_numpy()
        p = rd / f"{method}_PREDICTIONS_UNSCORED.csv"
        atomic_csv(p, pred)
        PREDICTION_JOBS.append(dict(panel=spec["panel"], seed=spec["seed"], budget=spec["budget"],
            n_source_questions=spec["n_source_questions"], n_source_rows=len(src), n_test_questions=spec["n_test_questions"],
            method=method, uses_probe_at_runtime=uses_probe, representation_dimensions=int(dimensions),
            path=str(p), policy_sha256=method_policy_hash or phash))

    save_predictions(predict_role_rule(rt, policy), "runtime_rule", False, 3)
    if RUN_RANDOM_HEAD_CONTROL:
        rp, _ = fit_role_rule(rsrc, ysrc, src, INPUTS.iloc[src].question,
            top_k=TOP_ROLE_HEADS, random_head_seed=spec["seed"] + 30000)
        rp["protocol_hash"] = PROTOCOL_HASH
        immutable_json(rd / "RANDOM_HEAD_POLICY_LOCK.json", rp)
        save_predictions(predict_role_rule(rt, rp), "random_heads_rule", False, 3, canonical_hash(rp))
    if RUN_SUPERVISED_DIAGNOSTICS:
        yh = ysrc.state.to_numpy(int)
        pred, model = fit_supervised_diagnostic(BLOCK[src], yh, BLOCK[test], spec["seed"])
        save_predictions(pred, "diagnostic_fullhidden_LR", True, 4096)
        del model
        uh = union_heads(policy)
        # Reference concatenates only these role-head LAST vectors; no reference k tuning.
        hsrc = HL[src][:, uh, :].reshape(len(src), -1)
        ht = HL[test][:, uh, :].reshape(len(test), -1)
        pred, model = fit_supervised_diagnostic(hsrc, yh, ht, spec["seed"])
        save_predictions(pred, "diagnostic_same_roleheads_LR", True, len(uh) * 128)
        del model
    audit = access.audit(); audit["n_unique_labeled_questions"] = INPUTS.iloc[audit["accessed_rows"]].question.nunique()
    assert audit["n_unique_labeled_rows"] <= len(src)
    assert audit["n_unique_labeled_questions"] == spec["n_source_questions"]
    atomic_json(rd / "LABEL_ACCESS_AUDIT.json", audit)
    print(f"[{si}/{len(SPECS)}] locked {spec['panel']} seed={spec['seed']} budget={spec['budget']:.0%}")
atomic_json(OUT / "PREDICTION_MANIFEST.json", PREDICTION_JOBS)
print("ALL FROZEN PREDICTIONS SAVED — next cell scores them.")

## 10. 이제 test 라벨을 평가 목적으로만 사용

예측·policy lock이 끝난 뒤 test K/C와 비교한다. 평가에서 정답 라벨을 쓰는 것은 label-free inference와 모순되지 않는다. **F1 자체가 label-free인 게 아니라 그 예측을 만드는 절차의 supervision 범위를 구분**해야 한다.

모든 네 class를 포함한 Macro-F1, K/C BAcc·AUROC, confusion matrix와 class별 F1을 저장한다. F1 신뢰구간은 해당 고정 정책의 test 질문 단위 bootstrap이며, source 재선정/발견의 전체 불확실성을 대신하지 않는다.

In [ ]:
# 10-1. 사후 평가. 학습/정책 파일은 변경하지 않음.
RESULT_ROWS = []
for job in PREDICTION_JOBS:
    pred = pd.read_csv(job["path"])
    ix = pred.row.to_numpy(int)
    y = LABELS.iloc[ix].state.to_numpy(int)
    met, cm = metrics_state(y, pred)
    lo, hi = cluster_f1_ci(y, pred.pred_state, pred.question, BOOTSTRAP_DRAWS, job["seed"] + 50000)
    chance = uniform_chance_reference(y, draws=200, seed=job["seed"] + 60000)
    r = {k: job[k] for k in ["panel", "seed", "budget", "n_source_questions", "n_source_rows",
                             "n_test_questions", "method", "uses_probe_at_runtime", "representation_dimensions"]}
    RESULT_ROWS.append({**r, **met, **chance, "F1_CI_low_conditional": lo, "F1_CI_high_conditional": hi})
    scored = pred.copy(); scored["true_state"] = y
    atomic_csv(Path(job["path"]).with_name(Path(job["path"]).name.replace("UNSCORED", "SCORED")), scored)
    pd.DataFrame(cm, index=STATE_NAMES, columns=STATE_NAMES).to_csv(
        Path(job["path"]).with_name(job["method"] + "_CONFUSION.csv"))
RESULTS = pd.DataFrame(RESULT_ROWS)
atomic_csv(OUT / "ALL_STATE_RESULTS.csv", RESULTS)
columns = ["panel", "seed", "budget", "method", "four_macro_F1", "K_AUROC", "C_AUROC", "uses_probe_at_runtime"]
display(RESULTS[columns])

SUMMARIES = []
for (panel, budget, method), g in RESULTS.groupby(["panel", "budget", "method"], sort=False):
    row = dict(panel=panel, budget=budget, method=method, seeds=len(g),
               n_source_questions=int(g.n_source_questions.iloc[0]), n_test_questions=int(g.n_test_questions.iloc[0]),
               uses_probe_at_runtime=bool(g.uses_probe_at_runtime.iloc[0]))
    for m in ["four_macro_F1", "four_BAcc_fixed4", "four_accuracy", "K_AUROC", "K_BAcc", "C_AUROC", "C_BAcc"]:
        row[m + "_mean"] = float(g[m].mean()); row[m + "_std_between_seeds"] = float(g[m].std(ddof=1)) if len(g)>1 else np.nan
    SUMMARIES.append(row)
SUMMARY = pd.DataFrame(SUMMARIES)
atomic_csv(OUT / "STATE_SUMMARY_BY_BUDGET.csv", SUMMARY)
print("주 방법: runtime_rule. diagnostic_* 점수는 supervised reference입니다.")
display(SUMMARY[["panel", "budget", "method", "four_macro_F1_mean", "four_macro_F1_std_between_seeds", "uses_probe_at_runtime"]])

In [ ]:
# 11. Head 반복 선택 / seed 간 overlap. 이전 E14 전체-data head 목록과의 일치를 선택 기준으로 쓰지 않음.
HEAD_FREQ, HEAD_JACCARD = head_stability(POLICY_RECORDS)
atomic_csv(OUT / "HEAD_SELECTION_FREQUENCY.csv", HEAD_FREQ)
atomic_csv(OUT / "HEAD_JACCARD_BY_SEED.csv", HEAD_JACCARD)
display(HEAD_FREQ.query("panel == 'primary_source20_test80' and budget == 0.2").sort_values(
    ["role", "selection_frequency", "head"], ascending=[True, False, True]).groupby("role").head(8))

In [ ]:
# 12. Label-efficiency curve. 기본 색상 사용; panel마다 별도 Figure.
import matplotlib.pyplot as plt
for panel in SUMMARY.panel.unique():
    fig, ax = plt.subplots(figsize=(9, 5))
    part = SUMMARY[SUMMARY.panel.eq(panel)]
    for method in ["runtime_rule", "random_heads_rule", "diagnostic_fullhidden_LR", "diagnostic_same_roleheads_LR"]:
        g = part[part.method.eq(method)].sort_values("budget")
        if g.empty: continue
        label = method + (" [supervised reference]" if method.startswith("diagnostic") else " [no runtime probe]")
        ax.errorbar(g.budget * 100, g.four_macro_F1_mean,
                    yerr=g.four_macro_F1_std_between_seeds.fillna(0), marker="o", capsize=3,
                    linestyle="--" if method.startswith("diagnostic") else "-", label=label)
    ax.set(xlabel="K/C labeled questions (% of all source-dataset questions)",
           ylabel="4-state Macro-F1: mean ± split-seed SD", title=panel, ylim=(0, 1))
    ax.legend(fontsize=8); ax.grid(alpha=.2); fig.tight_layout()
    fig.savefig(OUT / (panel + "_label_efficiency.png"), dpi=160)
    plt.show()

## 13. Online QA (선택 가능; 기본 켜짐) — source20 / seed42만

**목적:** 실제 규칙의 상태 예측이 QA 개선으로 연결되는지 별도 확인한다. 높은 state F1이 항상 높은 QA를 보장하지는 않는다. `K+C−→retain, K−C+→accept, 나머지→no-op`은 설계 가설이지 모든 샘플의 최적 개입 정답은 아니다.

비교: no-op / 새 4-state rule / 원래 E14 action rule / Always-retain / Always-accept / 같은 새 rule에 random direction. 이들 모두 동일한 source-only instruction vector와 **α=.5 + 5% norm cap**을 사용한다. 새 rule은 state만으로 action을 정하며 강도를 예측·학습하지 않는다.

Source의 최대 64개 질문에서 context/memory 지시문 차이로 vector를 **새로** 만든다. Target 질문·K/C/gold로 vector나 strength를 조정하지 않는다. 기존 E14의 대량-data vector는 재사용하지 않는다.

F1 분석이 기존 neutral bank를 재사용하는 것은 해당 bank가 입력에서 얻은 고정 activation이기 때문이다. QA는 결과표 lookup이 아니라 실제 한 번의 prefill 안에서 rule을 실행해 새 답변을 생성한다. 재실행용 batch cache는 코드/정책/데이터/vector fingerprint로 구분한다.

In [ ]:
# 13-1. Cache ↔ live runtime preflight — source-only, target tuning 없음.
# hidden provenance는 엄격히 확인하고, C_MASS는 raw 차이와 frozen-policy 결정 일치율을 함께 기록한다.
QA_AVAILABLE = bool(RUN_ONLINE_QA and torch.cuda.is_available())
if RUN_ONLINE_QA and not QA_AVAILABLE:
    print("GPU 없음: state 분석은 완료. Online QA는 미실행이며 성능을 만들어내지 않습니다.")
    atomic_json(OUT / "QA_STATUS.json", {"requested": True, "executed": False, "reason": "no GPU"})

if QA_AVAILABLE:
    engine = ensure_engine()
    primary_spec = next(s for s in SPECS if s["panel"] == "primary_source20_test80" and
                        s["seed"] == QA_SEEDS[0] and abs(s["budget"] - QA_BUDGET) < 1e-9)
    primary_policy = next(p for s, p in POLICY_RECORDS if
                          s["panel"] == primary_spec["panel"] and
                          s["seed"] == primary_spec["seed"] and
                          abs(s["budget"] - primary_spec["budget"]) < 1e-9)

    # source-only 32 rows를 실제 runtime batch size 그대로 다시 capture.
    check = np.asarray(primary_spec["source"][:min(32, len(primary_spec["source"]))], int)
    live_chunks = []
    for st in range(0, len(check), BATCH_SIZE):
        live_chunks.append(engine.capture(check[st:st+BATCH_SIZE]))
    live = {k: np.concatenate([z[k] for z in live_chunks], axis=0) for k in live_chunks[0]}
    cached = {
        "rows": check.copy(),
        "HEAD_Q": bank["HEAD_Q"][check].copy(),
        "HEAD_LAST": bank["HEAD_LAST"][check].copy(),
        "C_MASS": bank["C_MASS"][check].copy(),
        "BLOCK_LAST": bank["BLOCK_LAST"][check].copy(),
    }

    checks = {
        "n_source_rows_checked": int(len(check)),
        "thresholds_retuned": False,
        "target_labels_or_gold_used": False,
    }

    # 모델 revision / prompt / row order에 직접 민감한 hidden은 엄격히 검사.
    for name in ["HEAD_Q", "HEAD_LAST", "BLOCK_LAST"]:
        a = cached[name].astype(float).reshape(len(check), -1)
        b = live[name].astype(float).reshape(len(check), -1)
        cosine = np.sum(a*b, axis=1) / np.maximum(
            np.linalg.norm(a, axis=1) * np.linalg.norm(b, axis=1), 1e-12)
        rel = np.linalg.norm(a-b, axis=1) / np.maximum(np.linalg.norm(a, axis=1), 1e-12)
        checks[name] = {"min_cosine": float(cosine.min()),
                        "max_relative_l2": float(rel.max())}
        if cosine.min() < .999 or rel.max() > .03:
            atomic_json(OUT / "SOURCE_ONLY_RUNTIME_PREFLIGHT.json", checks)
            raise RuntimeError(
                "Hidden cache mismatch: " + name +
                ". prompt/revision/row order를 확인해야 합니다.")

    # C_MASS는 manual attention reconstruction + batch/padding 수치차이에 민감하므로
    # raw 0.005 하나만으로 전체 QA를 중단하지 않는다. 차이는 그대로 기록한다.
    c_abs = np.abs(cached["C_MASS"].astype(np.float32) - live["C_MASS"].astype(np.float32))
    checks["C_MASS"] = {
        "max_abs_error": float(c_abs.max()),
        "mean_abs_error": float(c_abs.mean()),
        "p95_abs_error": float(np.quantile(c_abs, .95)),
        "raw_exact_tolerance_0p005_pass": bool(c_abs.max() <= .005),
    }

    # frozen policy의 의미론적 결정이 유지되는지도 source-only로 확인.
    cached_rule = predict_role_rule(raw_features(cached), primary_policy)
    live_rule = predict_role_rule(raw_features(live), primary_policy)
    for col in ["pred_state", "state_action", "e14_action"]:
        neq = cached_rule[col].astype(str).to_numpy() != live_rule[col].astype(str).to_numpy()
        checks[col + "_mismatches"] = int(neq.sum())
        checks[col + "_agreement"] = float(1.0 - neq.mean())

    atomic_json(OUT / "SOURCE_ONLY_RUNTIME_PREFLIGHT.json", checks)
    print("Source-only live feature preflight:")
    print(json.dumps(checks, ensure_ascii=False, indent=2))

    if not checks["C_MASS"]["raw_exact_tolerance_0p005_pass"]:
        print(
            "주의: C_MASS raw 차이가 0.005를 넘었습니다. threshold를 target에 맞춰 재조정하지 않고 "
            "실제 online generation의 live pred_state F1 및 cache-vs-live mismatch를 별도 보고합니다.")

    print("PREFLIGHT PASS: hidden provenance 일치. C_MASS는 raw error + frozen-policy agreement로 기록.")


In [ ]:
# 13-2. Primary QA 조건들의 모든 답변을 먼저 저장. 아직 target gold를 읽지 않음.
QA_RUNS = []
if QA_AVAILABLE:
    for spec, policy in POLICY_RECORDS:
        if spec["panel"] != "primary_source20_test80" or spec["seed"] not in QA_SEEDS or abs(spec["budget"] - QA_BUDGET) > 1e-9:
            continue
        rd = Path(spec["root"]); qadir = rd / "online_QA"
        vec = build_source_vectors(engine, spec["source"], qadir / "vectors", spec["seed"], DIRECTION_QUESTIONS)
        rng = np.random.default_rng(spec["seed"] + 80000)
        random_vec = {}
        for l, v in vec.items():
            u = rng.standard_normal(len(v)).astype(np.float32)
            random_vec[l] = (u / np.linalg.norm(u) * np.linalg.norm(v)).astype(np.float32)
        phash = canonical_hash(json.loads((rd / "RUNTIME_POLICY_LOCK.json").read_text()))
        results = {}
        for mode in QA_MODES:
            usevec = random_vec if mode == "random_direction_state_rule" else vec
            vhash = hashlib.sha256(b"".join(np.asarray(usevec[l], np.float32).tobytes() for l in sorted(usevec))).hexdigest()
            fp = canonical_hash({"protocol": PROTOCOL_HASH, "policy": phash, "vector": vhash,
                                  "model": json.loads((OUT / "MODEL_LOCK.json").read_text())})
            results[mode] = generate_cached(engine, spec["test"], policy, usevec, mode, qadir / mode, fp, ALPHA, MAX_DELTA_RATIO)
        if "state_rule" in results and "random_direction_state_rule" in results:
            assert results["state_rule"].action.tolist() == results["random_direction_state_rule"].action.tolist()
        # Cached F1 rule and online rule must agree; a mismatch is reported, never tuned on target.
        prior = pd.read_csv(rd / "runtime_rule_PREDICTIONS_UNSCORED.csv").set_index("row")
        mismatch = int((results["state_rule"].set_index("row").pred_state.loc[spec["test"]].to_numpy() !=
                        prior.loc[spec["test"], "pred_state"].to_numpy()).sum())
        atomic_json(qadir / "CACHE_VS_LIVE_STATE_AGREEMENT.json", {"n": len(spec["test"]), "mismatches": mismatch,
                                                                 "thresholds_retuned": False})
        if mismatch: print("주의: cache vs live state mismatch", mismatch, "— online F1은 별도로 채점합니다.")
        QA_RUNS.append((spec, results))

In [ ]:
# 13-3. 실제 생성 후 gold/EM 평가. 상태 F1과 QA EM은 서로 다른 지표.
def normalize_answer(x):
    x = unicodedata.normalize("NFKC", str(x)).lower()
    x = "".join(c for c in x if c not in set(string.punctuation))
    return " ".join(w for w in x.split() if w not in {"a", "an", "the"})
def short_answer(x):
    x = str(x).strip().split("\n")[0].strip()
    x = re.sub(r"^(answer|the answer is)\s*:?\s*", "", x, flags=re.I)
    x = x.split(". ")[0].strip().strip("\"'")
    return re.sub(r"[.,;:\s]+$", "", x).strip()
def aliases(x):
    if isinstance(x, (list, tuple, np.ndarray)): a = list(x)
    else:
        try: a = json.loads(str(x))
        except Exception:
            try: a = ast.literal_eval(str(x))
            except Exception: a = [str(x)]
    if isinstance(a, str): a = [a]
    return [str(v) for v in a]
def grade(raw, row):
    p = normalize_answer(short_answer(raw))
    return int(any(p == normalize_answer(g) for g in aliases(QA_GOLD.iloc[int(row)].gold)))

QA_TABLES = []
for spec, generated in QA_RUNS:
    scored = {}
    for mode, pred in generated.items():
        df = pred.copy()
        df["em"] = [grade(t, r) for t, r in zip(df.raw, df.row)]
        atomic_csv(Path(spec["root"]) / "online_QA" / mode / "PREDICTIONS_SCORED.csv", df)
        scored[mode] = df
    base = scored["noop"].em.to_numpy(int)
    q = INPUTS.iloc[spec["test"]].question.to_numpy()
    state_true = LABELS.iloc[spec["test"]].state.to_numpy(int)
    for mode, df in scored.items():
        met = qa_pair_metrics(base, df.em, q, 2000, spec["seed"])
        met.update(seed=spec["seed"], budget=spec["budget"], method=mode,
                   intervention_rate=float(df.action.ne("noop").mean()),
                   max_delta_ratio=float(df.max_applied_delta_ratio.max()),
                   seconds_per_row_amortized=float(df.seconds_per_row_amortized.mean()),
                   hit_token_limit_rate=float(df.hit_token_limit.mean()))
        delta = df.em.to_numpy(int) - base
        for sid, name in enumerate(STATE_NAMES):
            ix = state_true == sid
            met[name + "_DeltaEM"] = float(delta[ix].mean()) if ix.any() else np.nan
        if "always_retain" in scored:
            comp = qa_pair_metrics(scored["always_retain"].em, df.em, q, 2000, spec["seed"]+1)
            met.update(vs_always_retain_DeltaEM=comp["DeltaEM"],
                       vs_always_retain_CI_low=comp["DeltaEM_CI_low"], vs_always_retain_CI_high=comp["DeltaEM_CI_high"])
        if "pred_state" in df:
            pm, cm = metrics_state(state_true, df)
            met["online_four_macro_F1"] = pm["four_macro_F1"]
        QA_TABLES.append(met)
QA_RESULTS = pd.DataFrame(QA_TABLES)
if len(QA_RESULTS):
    atomic_csv(OUT / "ONLINE_QA_SUMMARY.csv", QA_RESULTS)
    atomic_json(OUT / "QA_STATUS.json", {"requested": True, "executed": True, "seeds": QA_SEEDS, "budget": QA_BUDGET})
    display(QA_RESULTS)
else:
    print("Online QA 결과 없음 (옵션 꺼짐 또는 GPU 없음). F1 결과와 혼동하지 않습니다.")

## 14. 외부 데이터셋에 frozen policy 적용 (데이터가 있을 때만)

아래 함수는 **새로 fit하지 않는다.** 소스 JSON에 저장된 head/sign/normalizer/threshold를 그대로 사용한다. 외부 bank는 같은 backbone/revision·Q-first prompt·layer/head 정의로 미리 추출해야 한다. Target의 전체 분포를 보고 quantile을 다시 맞추지 않는다.

`EXTERNAL_LABELS_PATH`는 선택 사항이다. 없으면 prediction만 저장한다. 있으면 예측 저장 후에만 F1을 계산한다. 새 데이터의 K 라벨도 같은 모델의 고정 closed-book 프로토콜로, C 라벨도 같은 정의로 구성되어야 비교 가능하다. K는 운영상 정의한 정오 proxy이며 모델의 모든 지식을 완벽하게 판정하는 의미는 아니다.

In [ ]:
# 14-1. 새 데이터가 없으면 아무 결과도 만들지 않음.
def predict_external_bank(policy_json_path, bank_npz_path, inputs_csv_path, output_csv_path):
    policy = json.loads(Path(policy_json_path).read_text())
    with np.load(bank_npz_path, allow_pickle=False) as z: ext = {k: z[k].copy() for k in z.files}
    ext_inputs = pd.read_csv(inputs_csv_path)
    assert set(["question", "ctx_text"]).issubset(ext_inputs.columns)
    assert np.array_equal(ext["rows"], np.arange(len(ext_inputs)))
    assert ext["HEAD_Q"].shape[1:] == (32, 128) and ext["HEAD_LAST"].shape == ext["HEAD_Q"].shape
    pred = predict_role_rule(raw_features(ext), policy)
    pred.insert(0, "row", ext["rows"])
    pred["question"] = ext_inputs.question.astype(str).to_numpy()
    atomic_csv(output_csv_path, pred)  # gold 파일을 열기 전에 저장
    return pred

if EXTERNAL_BANK_PATH and EXTERNAL_INPUTS_PATH:
    source_spec = next(s for s in SPECS if s["panel"] == "primary_source20_test80" and s["seed"] == 42 and abs(s["budget"]-.20)<1e-9)
    pred = predict_external_bank(Path(source_spec["root"])/"RUNTIME_POLICY_LOCK.json", EXTERNAL_BANK_PATH,
                                 EXTERNAL_INPUTS_PATH, OUT/"EXTERNAL_PREDICTIONS_UNSCORED.csv")
    print("Frozen external predictions saved; no target fitting.")
    if EXTERNAL_LABELS_PATH:
        gt = pd.read_csv(EXTERNAL_LABELS_PATH).sort_values("row")
        assert np.array_equal(gt.row.to_numpy(), pred.row.to_numpy())
        met, cm = metrics_state(state_ids(gt.k, gt.c), pred)
        atomic_json(OUT/"EXTERNAL_STATE_METRICS.json", met)
        display(pd.DataFrame([met]))
else:
    print("외부 dataset은 제공되지 않았습니다. 이번 결과는 p8s 내부 low-label 실험입니다.")

In [ ]:
# 15. 자동 보고서. 주 방법/참고 probe/QA를 분리해서 기록.
QA_RESULTS = globals().get("QA_RESULTS", pd.DataFrame())
primary = RESULTS[(RESULTS.panel == "primary_source20_test80") & (RESULTS.method == "runtime_rule")]
lines = ["# E16 Low-Label E14 결과", "", "## 주 방법",
         "Source K/C labels로 head/sign을 발견하고, source activation quantile을 고정한 probe-free rule.",
         "Utility predictor 및 per-sample strength learning 없음.", "",
         "## 실제 규칙 4-state F1 (반복 hold-out, OOF 합산 아님)",
         "| Budget | Source questions | Test questions | Mean Macro-F1 | Seed SD |", "|---:|---:|---:|---:|---:|"]
for b, g in primary.groupby("budget"):
    lines.append(f"|{b:.0%}|{int(g.n_source_questions.iloc[0])}|{int(g.n_test_questions.iloc[0])}|{g.four_macro_F1.mean():.4f}|{g.four_macro_F1.std(ddof=1):.4f}|")
lines += ["", "## 해석 범위",
          "- diagnostic_*_LR은 source 라벨로 학습한 supervised reference. 주 rule 성능이 아님.",
          "- source-only 50th-percentile K/C decoder는 이번 확장. 기존 E14의 0.6306을 기대값으로 가정하지 않음.",
          "- state→action 규칙은 설계 가설. 실제 이득/손상은 별도 QA 결과로 평가.",
          "- 동일 p8s 내부 실험이며, 5개 seed 중 가장 좋은 것만 택하지 않음.",
          "- source discovery도 라벨을 썼으므로 전체 방법은 fully label-free가 아님.",
          "- 공개한 p8s 결과를 보고 다음 설계를 바꾸면 연구 수준에서 적응적 개발이 됨. 최종 확인은 새 외부 데이터가 필요.",
          "", "## 온라인 QA", "실행 결과: ONLINE_QA_SUMMARY.csv" if len(QA_RESULTS) else "미실행. 성능 결과 없음.",
          "", "## 주요 산출물", "STATE_SUMMARY_BY_BUDGET.csv / ALL_STATE_RESULTS.csv / HEAD_SELECTION_FREQUENCY.csv / HEAD_JACCARD_BY_SEED.csv",
          "각 run의 RUNTIME_POLICY_LOCK.json / LABEL_ACCESS_AUDIT.json / *_PREDICTIONS_UNSCORED.csv / *_CONFUSION.csv",
          "", f"Protocol SHA256: `{PROTOCOL_HASH}`"]
report = "\n".join(lines)
(OUT/"REPORT.md").write_text(report, encoding="utf-8")
print(report)
print("\n완료. 결과 폴더:", OUT)

## 16. 판정 기준과 다음 해석

**주 rule F1이 소량 source에서도 유지:** 적은 supervision으로 role 신호를 고정해 held-out 4-state를 읽을 수 있다는 근거. 여전히 새 데이터셋 전이는 별도 검증해야 한다.

**Reference probe는 좋은데 rule F1만 낮음:** 선택 head에 읽을 수 있는 정보는 있지만 q-norm/mass/median threshold가 충분히 읽지 못할 가능성. “head에 신호가 없다” 또는 “threshold만 바꾸면 된다”로 단정하지 않는다.

**Rule F1은 괜찮지만 QA가 약함:** 상태 판별과 실제 actuator의 효과는 별개다. Utility 학습을 자동 추가하지 않는다. 고정 vector 자체의 효과, 각 상태별 Recovery/Damage를 먼저 분리해서 본다.

**주의:** median threshold는 확률 .5가 아니다. source score의 중앙값이며, 실제 K+ 비율이나 정확도를 최적으로 맞추는 경계가 아니다. 이 선택도 평가 대상이다. 결과가 낮아도 source/test 규칙을 사후 변경하지 않는다.

### 구현 참고 (실험 주장과 분리)
- 원본 E14 notebook: `discover_heads`, `build_role_policy`, `L13Capture`, `HeadGateSession`, `build_fold_steering`.
- scikit-learn [Data leakage](https://scikit-learn.org/stable/common_pitfalls.html#data-leakage): preprocessing/선택은 source 안에서만 fit.
- Transformers [v4.57.1 KV cache](https://huggingface.co/docs/transformers/v4.57.1/kv_cache): prefill 후 같은 cache로 decode.

**검증 상태:** 제작 환경에서는 실제 p8s neutral bank와 A100/Llama 가중치에 접근하여 실험을 실행하지 않았다. 코드 문법 및 합성 데이터 기반 numerical/누수 방지 테스트 결과는 별도 제작 검증 파일에 기록한다. 실제 F1/QA 수치는 이 notebook을 Colab에서 실행한 뒤에만 나온다.